# 01 — Khám phá dữ liệu
Notebook này giúp bạn:
- Kiểm tra cấu trúc dataset
- Xem ảnh mẫu từng loại cây
- Kiểm tra phân phối số lượng ảnh
- Xem kết quả data augmentation

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import random

from config import CLASSES, CLASS_INFO, DATA_DIR
from utils.dataset import dataset_stats, create_generators

print('Setup xong!')

## 1. Thống kê dataset

In [ ]:
stats = dataset_stats(verbose=True)

## 2. Xem ảnh mẫu từng loại cây

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Ảnh mẫu từng loại cây lương thực', fontsize=14, fontweight='bold')
for ax, cls in zip(axes.flat, CLASSES):
    imgs = list((DATA_DIR / 'train' / cls).glob('*.*'))
    if imgs:
        img = Image.open(random.choice(imgs)).convert('RGB')
        ax.imshow(img)
        info = CLASS_INFO[cls]
        ax.set_title(f"{info['emoji']} {info['vi']}", fontsize=11)
    else:
        ax.text(0.5, 0.5, 'Chưa có ảnh', ha='center', va='center', transform=ax.transAxes)
    ax.axis('off')
plt.tight_layout(); plt.show()

## 3. Biểu đồ phân phối số lượng ảnh

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, split in zip(axes, ['train', 'val', 'test']):
    names  = [CLASS_INFO[c]['vi'] for c in CLASSES]
    counts = [stats[split].get(c, 0) for c in CLASSES]
    ax.barh(names, counts, color='#1976D2')
    ax.set_title(f'Tập {split} ({sum(counts)} ảnh)')
    ax.set_xlabel('Số ảnh')
plt.tight_layout(); plt.show()

## 4. Data Augmentation

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
train_gen, _, _ = create_generators(augment=True)

X, y = next(train_gen)
fig, axes = plt.subplots(2, 8, figsize=(20, 6))
fig.suptitle('Ảnh sau augmentation (mỗi hàng 8 ảnh)', fontsize=12)
for i, ax in enumerate(axes.flat):
    if i < len(X):
        ax.imshow(X[i])
    ax.axis('off')
plt.tight_layout(); plt.show()

## 5. Kiểm tra sau khi train

In [ ]:
# Chạy sau khi đã train xong
import tensorflow as tf
from config import BEST_MODEL_PATH
try:
    model = tf.keras.models.load_model(str(BEST_MODEL_PATH))
    model.summary()
except FileNotFoundError:
    print('Chưa có model. Chạy: python ../train.py')